# **HOMEWORK 2 - COMPUTER SECURITY**

**Mateo Vivanco (00328476)** <br>
**Roberto Villafuerte (00329429)**

# **Architecting the AutoDrive Firmware Pipeline**

This system relies on a **Hybrid Cryptographic Design**. Asymmetric encryption (RSA) is mathematically intense and far too slow to encrypt a 2GB file. Symmetric encryption (AES) is lightning-fast but suffers from the "Key Distribution Problem" (how to safely share the key). 

By combining them, we use RSA to securely exchange a temporary AES key, and use that AES key to encrypt the heavy lifting.

---

## **Part 1: RSA Key Generation (The Mechanics)**

RSA relies on the mathematical difficulty of prime factorization. Every vehicle must generate its own unique key pair before leaving the factory.

### 1. Manual Calculation
Let's walk through the exact algorithm using the assigned prime numbers.

* **Step 1: Choose Primes**
    $$p = 11, q = 13$$
* **Step 2: Calculate the Modulus (n)**
    This forms the maximum value for the math and is part of both keys.
    $$n = p * q = 11 * 13 = 143$$
* **Step 3: Calculate Euler's Totient ($\varphi(n)$)**
    This calculates the count of numbers that are coprime to n.
    $$\varphi(n) = (p - 1) \cdot (q - 1) = 10 \cdot 12 = 120$$
* **Step 4: Choose the Public Exponent (e)**
    Must be an integer between 1 and 120 that is coprime to 120. Let's choose 7.
    $$e = 7$$
* **Step 5: Calculate the Private Exponent (d)**
    We need the modular multiplicative inverse of e mod φ(n). We need a value for d where (e * d) divided by 120 leaves a remainder of 1.
    $$e \cdot d \equiv 1 \pmod{\varphi(n)}$$
    $$7 \cdot d \equiv 1 \pmod{120}$$
    $\implies d = 103$, because $7 \cdot 103 = 721$, and $721 \pmod 120 = 1$.

**The Final Keys:**
* **Public Key:** $(e=7, n=143)$
* **Private Key:** $(d=103, n=143)$

#### 1.1. Verification of encryption and decryption

- **Step 1: Encryption (Using the Public Key)**

Anyone can encrypt a message using the Public Key. The mathematical formula for RSA encryption is:
$$C = M^e \pmod{n}$$
*(Where $C$ is the Ciphertext, $M$ is the Message, $e$ is the Public Exponent, and $n$ is the Modulus).*

Let's plug in our numbers:
1.  $C = 42^7 \pmod{143}$
2.  Calculate the exponent: $42^7 = 230,539,333,248$
3.  Apply the modulo (divide by 143 and find the remainder):
    * $230,539,333,248 \div 143 = 1,612,163,169$ with a remainder of **$81$**.

The encrypted ciphertext is **`81`**. This is what is transmitted over the network. If a hacker intercepts `81`, it looks like garbage data to them.

- **Step 2: Decryption (Using the Private Key)**

To read the ciphertext `81`, the receiver must use their Private Key ($d=103$). The formula for decryption is:
$$M = C^d \pmod{n}$$

Let's plug in the numbers:
1.  $M = 81^{103} \pmod{143}$
2.  Calculate the exponent: $81^{103}$ is a staggeringly massive number (it has nearly 200 digits). 
3.  Apply the modulo: When computers do this, they don't actually calculate the 200-digit number; they use an algorithm called "modular exponentiation" to find the remainder efficiently. 
4.  If you divide that massive number by 143, the exact remainder left over is **`42`**.

The receiver successfully recovers the original message of **`42`**.

### 2. Implementation Logic (2048-bit Security)
In the real world, $p$ and $q$ are randomly generated primes that are each over $300$ digits long. The modulus $n$ becomes a 617-digit number. 

If a hacker intercepts the public key, they know $n$ and $e$. To find the private key $d$, they must calculate the totient $\varphi(n)$. To calculate the totient, they **must** know the original primes $p$ and $q$. Factoring a 617-digit number back into its two primes requires more computational time than the current age of the universe using classical computing. This is why RSA is secure.

To understand why a hacker cannot simply calculate the private key ($d$) from the public key ($n$ and $e$), we have to look at the strict mathematical sequence required to generate $d$, and where that sequence creates an impassable bottleneck.

This concept relies on what cryptographers call a **Trapdoor One-Way Function**—a mathematical operation that is incredibly easy to do in one direction, but practically impossible to reverse unless you possess a specific piece of secret information.

#### 2.1. The Mathematical Trapdoor (The Attacker's Dilemma)
Imagine an attacker intercepts the AutoDrive public key. They now possess two numbers:
* The Modulus ($n$)
* The Public Exponent ($e$)

The attacker's goal is to find the Private Exponent ($d$). They know the exact formula used to create $d$:
$$e \cdot d \equiv 1 \pmod{\phi(n)}$$

To solve this algebraic equation for $d$, the attacker **must** know the value of Euler's Totient, $\phi(n)$. There is no known shortcut around this requirement in classical mathematics. 

#### 2.2. The Bottleneck (Calculating the Totient)
The attacker knows that $n$ was created by multiplying two prime numbers, $p$ and $q$. They also know the formula for the totient:
$$\phi(n) = (p - 1) \cdot (q - 1)$$

Here lies the cryptographic trap: **To calculate $\phi(n)$, you absolutely must know the original values of $p$ and $q$.** Since the attacker only has $n$, they must work backward. They must take the massive number $n$ and figure out which two prime numbers were multiplied together to create it. This is known as the **Integer Factorization Problem**.

#### 2.3. The Scale of 2048-bit Keys
In our toy example, $n$ was $143$. It takes a human seconds to realize that $11 \times 13 = 143$. 

In a real-world **RSA-2048** system, the key length refers to the size of the modulus $n$ in binary bits. A 2048-bit number translates to roughly **617 decimal digits**. 
* The original primes, $p$ and $q$, are each about 300 digits long.
* To put this scale into perspective, the number of atoms in the observable universe is estimated to be roughly $10^{80}$ (an 81-digit number). The RSA modulus $n$ is a $10^{617}$ number. It is unfathomably huge.

#### 2.4. Computational Asymmetry (Multiplication vs. Factorization)
This is where the "computationally impossible" aspect comes into play. 

**The Forward Operation (Easy):**
If the AutoDrive server wants to generate a key, it randomly selects two 300-digit prime numbers and multiplies them together. A standard laptop processor can multiply two 300-digit numbers in a fraction of a millisecond.

**The Reverse Operation (Hard):**
If a hacker wants to factor the resulting 617-digit number back into its original two primes, they cannot just "un-multiply" it. They must use advanced algorithms, the most efficient of which is currently the **General Number Field Sieve (GNFS)**. 

Even using the GNFS, the time required to factor a 2048-bit number scales exponentially. 
* As of today, the largest RSA number ever successfully factored by humanity was 829 bits long (RSA-250). It took thousands of computers running in parallel for thousands of hours to crack.
* Cracking a 2048-bit number is not just twice as hard as cracking a 1024-bit number; it is billions of times harder. 

It is estimated that factoring a 2048-bit RSA modulus using every classical computer currently on Earth would take millions of years. Therefore, because the hacker cannot factor $n$, they cannot find $p$ and $q$. Because they lack $p$ and $q$, they cannot calculate $\varphi(n)$. And without $\varphi(n)$, the private key $d$ remains locked behind an unbreakable wall of mathematics.

---

## **Part 2: Cryptographic Hashes (Integrity Check)**

Hashes act as the digital fingerprint of the 2GB firmware file. 

### 1. The Architecture of SHA-256 (Handling the 2GB Payload)
How does an algorithm turn a 2GB file into a fixed 256-bit string without losing the uniqueness of the data? It uses a process called the **Merkle-Damgård construction**.

* **Chunking:** The algorithm doesn't read the 2GB file all at once. It breaks the file down into tiny **512-bit blocks**.
* **The State:** SHA-256 maintains a running "state" of 256 bits (stored in 8 separate 32-bit registers).
* **The Compression Function:** The algorithm takes the first 512-bit block of the firmware, mixes it with the initial state, and compresses it down to a new 256-bit state. It then takes the *next* 512-bit block, mixes it with that *new* state, and compresses it again. 
* This chain continues for all 33 million blocks in the 2GB file. The final 256-bit state after the last block is processed becomes the final hash.

### 2. The Avalanche Effect
Cryptographic hash functions like SHA-256 are explicitly designed to be highly non-linear. If you flip exactly **one single bit** in a 2GB file (which contains $17,179,869,184$ bits), approximately **50%** (roughly $128$ bits) of the bits in the resulting 256-bit hash will flip. 

* **Message Expansion:** When a 512-bit block is processed, SHA-256 expands it into an array of 64 distinct "words." If you flip just one bit in the original file, that bit gets multiplied and scattered across this entire 64-word array.
* **The 64 Rounds of Chaos:** SHA-256 then runs the data through 64 rounds of heavy mathematical transformation. It uses non-linear functions (like `Maj` for majority and `Ch` for choose), bitwise XORs, and right-rotations. 
* **The Ripple Effect:** By round 3 or 4, that single flipped bit has altered several bits. By round 20, it has altered dozens. By round 64, the mathematical chaos is absolute. Furthermore, because of the Merkle-Damgård chain mentioned above, this completely randomized state is fed into the next block, meaning the rest of the 2GB file is now being mixed with entirely different math.
* **MITM Implication:** If a hacker intercepts the 2GB download and changes a single "0" to a "1" to alter an engine parameter, the resulting hash will look completely unrecognizable. The car will compare this mutant hash to the official signature from HQ, see they don't match, and immediately abort the update.

### 2. Collision Resistance
A collision happens when two mathematically distinct files generate the exact same hash output. Why does a collision break the entire security system?

**Why does a collision break the entire security system?**
* **The Vulnerability:** The RSA Digital Signature from AutoDrive HQ does *not* encrypt the firmware; it only encrypts the *hash*. 
* **The Attack:** If a hacker creates a malicious `Virus.bin` that hashes to the exact same 256-bit string as the official `Update.bin`, they can swap the files on the network. The car will download `Virus.bin`, hash it, and get `Hash_X`. It will then decrypt HQ's digital signature and also get `Hash_X`. The car's logic dictates: "The hashes match, therefore this file is official." The car installs the virus.

**Why is finding a collision considered computationally impossible?**
* The output space of SHA-256 is $2^{256}$. This is a number roughly equal to the amount of atoms in the known universe.
* However, because of a statistical phenomenon called the **Birthday Paradox**, attackers don't need $2^{256}$ guesses to find two matching hashes. They only need the square root of the output space, which is $2^{128}$ guesses. 
* Even with this statistical shortcut, $2^{128}$ operations is so astronomically large that if every computer on Earth guessed a billion hashes per second, the sun would burn out before they found a single SHA-256 collision. This mathematical vastness is what guarantees the integrity of the AutoDrive system.

---

## **Part 3: Digital Certificates (The Chain of Trust)**

How does the car know that the "AutoDrive HQ Public Key" actually belongs to AutoDrive, and not a hacker spoofing the server? It uses a Certificate Authority (CA).

In the AutoDrive scenario, we established that RSA and AES keep the firmware secure, and SHA-256 proves it wasn't corrupted. However, a massive vulnerability remains: **Authentication.** If a hacker intercepts the car's connection to the internet, they can execute a Man-in-the-Middle (MITM) attack. The hacker tells the car, *"Hi, I am AutoDrive HQ. Here is my Public Key."* If the car blindly accepts this key, the hacker can send malicious firmware, properly encrypt it, properly hash it, and sign it with their own rogue Private Key. The car's math will check out perfectly, and it will install the virus.

### 1. The Certificate Authority (The Anchor of Trust)

To prevent this, we cannot trust raw Public Keys. We must wrap them in **Digital Certificates** verified by a **Certificate Authority (CA)**.

A Certificate Authority (like DigiCert, IdenTrust, or Let's Encrypt) is a globally recognized, highly secure organization. Their sole job is to verify the real-world identity of companies and issue cryptographic proof of that identity. 

They are the digital equivalent of a government passport office. You trust a passport because you trust the government that issued it. The car trusts the AutoDrive Public Key because it trusts the CA that signed it.

### 2. Certificate Signing Request (CSR)
AutoDrive HQ must generate a CSR and send it to a trusted CA (like DigiCert). The CSR contains:
1.  **Subject Identity:** Organization Name (AutoDrive Inc.), Locality, State, Country.
2.  **Subject Public Key:** The actual RSA Public Key HQ wants to use.
3.  **Signature Algorithm:** The hashing and encryption algorithm used (e.g., `sha256WithRSAEncryption`).
4.  **Self-Signature:** HQ signs the CSR with its own Private Key to mathematically prove to the CA that it actually controls the matching Private Key.

Before AutoDrive can send updates, it must prove its identity to the CA using a **Certificate Signing Request (CSR)**.

1. **Generation:** AutoDrive HQ generates its RSA Key Pair (Public and Private).
2. **The Payload:** HQ creates a text file containing:
   * Their Organization Name ("AutoDrive Inc.")
   * The specific domain they will use ("updates.autodrive.com")
   * Their newly generated **Public Key**.
3. **The Proof of Possession:** HQ mathematically signs this CSR with their own **Private Key**. This proves to the CA that whoever is submitting the request actually controls the Private Key matching the Public Key inside the CSR.
4. **The Audit:** AutoDrive sends the CSR to the CA. The CA performs real-world checks (e.g., verifying AutoDrive's corporate registration, calling their executives, proving they own the "autodrive.com" domain name).

### 3. Validation via Root Certificates
Once the CA verifies AutoDrive is a real company, they issue an X.509 Certificate. For the car to validate this certificate, it must possess the **Root Certificate of the CA**. This Root Certificate contains the CA's Public Key and must be hardcoded into the car's read-only memory at the factory. 

### 4. The X.509 Certificate (The Final Output)
Once the CA is satisfied that AutoDrive is legitimate, they issue an **X.509 Digital Certificate**. 

This certificate contains AutoDrive's Public Key and identity information, but the crucial component is the **CA's Digital Signature**. The CA hashes the entire certificate and encrypts that hash using the **CA's own Private Key**. 

This document is now cryptographically bound. If a hacker intercepts AutoDrive's certificate and tries to replace AutoDrive's Public Key with their own rogue key, the CA's digital signature will instantly break, rendering the certificate invalid.

### 5. The Chain of Trust (How the Car Verifies the Certificate)
When the car connects to download the firmware, HQ sends its X.509 Certificate instead of just a raw Public Key. The car must now verify it. This is where the **Chain of Trust** operates.

**Step 1: The Root Store**
When the car was manufactured, engineers hardcoded a "Root Store" into its unalterable memory. This is a list of Root Certificates belonging to the world's top Certificate Authorities. These Root Certificates contain the **Public Keys of the CAs**. 

**Step 2: Signature Decryption**
The car receives AutoDrive's certificate. It sees that it was signed by "DigiCert." The car looks into its Root Store, finds the DigiCert Root Certificate, and extracts DigiCert's Public Key. It uses this CA Public Key to decrypt the signature attached to AutoDrive's certificate.

**Step 3: The Hash Comparison**
The car then calculates its own hash of AutoDrive's certificate data. It compares its local hash to the hash it just decrypted from the CA's signature. 
* If they **do not match** (or if the signature fails to decrypt), it means the certificate is forged, tampered with, or signed by an untrusted entity. The car terminates the connection.
* If they **match perfectly**, the car knows with absolute mathematical certainty that a trusted CA vouched for this specific Public Key.

Only after this Chain of Trust is validated will the car extract the AutoDrive HQ Public Key from the certificate and use it to verify the 2GB firmware payload.

---

## **Part 4: Digital Signatures (Authentication)**

If encryption is about hiding information, digital signatures are about proving its origin. In the AutoDrive pipeline, the car must know with absolute mathematical certainty that the firmware came from AutoDrive HQ.

Digital Signatures apply RSA in reverse. Instead of encrypting with the Public Key so only the Private Key can read it, you encrypt with the Private Key so *anyone* with the Public Key can verify you wrote it.

### 1. The Mathematical Constraint (Why we hash before signing)
A critical limitation of RSA cryptography is its **block size limit**. A 2048-bit RSA key can only encrypt a maximum of 256 bytes of data at a single time. 

Because the firmware is 2GB, it is mathematically impossible to encrypt it directly with RSA. Even if we broke the 2GB file into millions of 256-byte chunks and encrypted each one with RSA, the computational overhead would be so massive that the car would take weeks to decrypt the update.

**The Solution:** We sign the *hash*, not the file. 
SHA-256 reduces the massive 2GB file down to a tiny 32-byte (256-bit) string. This 32-byte string easily fits inside RSA's 256-byte limit.

### 1. The Signing Workflow
Normally, RSA is used for confidentiality: you encrypt with the receiver's Public Key, so only their Private Key can read it. 

Digital Signatures flip this logic. AutoDrive HQ wants *everyone* (all the cars) to be able to read the signature, but they want to prove that *only HQ* could have created it.
1.  **Creation:** HQ hashes the 2GB firmware. They then encrypt that hash using the **AutoDrive HQ Private Key**. 
2.  **Verification:** The car receives the signed hash. It uses the **AutoDrive HQ Public Key** to decrypt it. 
3.  **The Proof:** If the signature successfully decrypts using HQ's Public Key into a valid SHA-256 hash, the car knows definitively that the data must have been encrypted by the corresponding Private Key. Since only HQ possesses that Private Key, the file is authentic.

### 2. Non-Repudiation (The Legal Cryptography)
"Repudiation" is the act of denying you did something. In cybersecurity, **Non-Repudiation** means a system provides irrefutable proof of an action, to the point that it would hold up in a court of law.

If a faulty update causes an AutoDrive car to crash, the company might try to claim a hacker spoofed the update. Cryptography destroys this defense. Because the update was signed by the HQ Private Key—and HQ is the sole entity on Earth that possesses that key—the math serves as undeniable proof that HQ authorized the payload.

Non-repudiation guarantees that the sender cannot legally or technically deny sending the message. Because AutoDrive HQ is the sole possessor of the HQ Private Key, it is mathematically impossible for anyone else (a hacker, a rogue employee, a competitor) to have generated that specific encrypted hash. 

---

## **Part 5: Secure Update Protocol (The Workflow)**

We now have all the individual cryptographic primitives:
* **RSA:** Highly secure, solves the key distribution problem, but is incredibly slow and has severe data size limits.
* **AES:** Lightning-fast, can encrypt terabytes of data effortlessly, but requires both parties to safely share the exact same secret key.
* **SHA-256:** Proves data integrity but doesn't hide data or prove who sent it.

To build the ultimate pipeline, we combine them into a **Hybrid Cryptosystem**. We use the slow, secure math (RSA) strictly to exchange a temporary key and verify identities, and use the fast math (AES) to encrypt the heavy 2GB payload.

Here is the step-by-step hybrid protocol flow between HQ and the Vehicle.

#### Phase 1: Preparation, Encryption & Packaging (AutoDrive HQ)
1.  HQ finalizes the `Firmware.bin` (2GB) file.
2.  **Symmetric Key Generation:** HQ generates a completely random, single-use 256-bit **AES Session Key**.
3.  **Confidentiality:** HQ encrypts `Firmware.bin` using the AES Session Key.
4.  **Integrity & Authentication:** HQ hashes the unencrypted `Firmware.bin` using SHA-256. HQ then encrypts this hash with the **HQ Private Key** to create the Digital Signature.
5.  **Secure Key Exchange:** HQ takes the raw AES Session Key and encrypts it using the specific **Car's Public Key**.

#### Phase 2: Transmission
HQ bundles the pieces together and sends the payload over the internet:
`[Encrypted AES Key] + [Digital Signature] + [AES-Encrypted Firmware]`

#### Phase 3: Unpacking & Execution (The Vehicle)
When the car receives the bundle, it executes the cryptography in reverse.
1.  **Extract the Key:** The car uses its own **Car Private Key** to decrypt the AES Session Key.
2.  **Decrypt the Payload:** The car uses the newly decrypted AES Session Key to decrypt the 2GB firmware file.
3.  **Calculate Local Integrity:** The car runs the decrypted firmware through SHA-256 to generate its own local hash.
4.  **Verify the Signature:** The car uses the **HQ Public Key** (which it previously verified via the Root CA) to decrypt the Digital Signature, revealing HQ's original hash.
5.  **The Final Check:** The car compares the local hash to HQ's decrypted hash. If they match perfectly, the car installs the update.

### Requirement Validation Matrix

| Design Requirement | Protocol Mechanism Used to Achieve It |
| :--- | :--- |
| **1. Secure Key Exchange** | The fast AES Session Key is wrapped in the **Car's Public Key**. It can only be extracted by the car's local hardware. |
| **2. Confidentiality** | The massive 2GB payload is wrapped in **AES-256 symmetric encryption**, rendering it unreadable to anyone sniffing the network. |
| **3. Integrity** | The car calculates a local **SHA-256 hash** of the decrypted firmware. If the file was corrupted during download, the final hash comparison will fail. |
| **4. Authentication** | The signature is decrypted using the **HQ Public Key** (verified via the Root CA). If it decrypts cleanly, it definitively originated from AutoDrive HQ. |
| **5. Non-Repudiation** | The signature was created using the **HQ Private Key**. HQ cannot claim a hacker fabricated the update, as the math proves HQ's keys were used. |

---

### **AI Utilization Statement**
*The cryptographic concepts, mathematical logic, and system architecture design presented in this assignment represent the original work and understanding of the authors. Generative AI tools were utilized strictly as a writing assistant to refine prose, enhance grammatical clarity, and structure the formatting for improved readability.*

---